# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohmasaeed/flyrank-ml-internship-/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Which pages should be reviewed first for refresh, expansion, protection,
pruning, or monitoring based on observed search and engagement signals?

### Decision Supported

The goal is to build a repeatable content-opportunity scoring system that
helps prioritize pages for human review.

The system is intended as decision support. It does not claim that a
recommended refresh will cause future performance improvement.

### Lane

This capstone follows Lane 2: Refresh / Content Opportunity Scoring.

The analysis uses historical page-level performance signals to identify
pages that show evidence of deterioration, growth, recovery, or other
review-worthy patterns.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Install packages**

In [3]:
!pip install -q duckdb huggingface_hub pyarrow pandas scikit-learn matplotlib

In [4]:
from huggingface_hub import login

login()

**DuckDB connection**

In [5]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully.")

DuckDB connected successfully.


**Dataset path**

In [6]:
HF_DATASET = "hf://datasets/FlyRank/internship-warehouse"

print(HF_DATASET)

hf://datasets/FlyRank/internship-warehouse


In [7]:
import os
from huggingface_hub import get_token

token = get_token()

print("Token available:", token is not None)

Token available: True


In [8]:
con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{token}'
)
""")

print("Hugging Face authentication configured for DuckDB.")

Hugging Face authentication configured for DuckDB.


**dim_content inspect**

In [9]:
content_sample = con.sql(f"""
    SELECT *
    FROM read_parquet('{HF_DATASET}/dim_content.parquet')
    LIMIT 5
""").df()

content_sample

,client_hash_id,content_hash_id,keyword_hash_id,url_hash_id,keyword_char_count,keyword_token_count,url_char_count,content_created_date,content_updated_date,content_type,...,category_count,keyword_created_date,provider_used,model_used,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword_e754999ab88dd9f2,url_d6091f18cf628794,22,4,108,2026-05-30,2026-07-01,keyword article,...,3,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword_4329d7aede8e208b,url_3a66d2f2e36823ca,31,6,95,2026-06-12,2026-07-01,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword_9b08047d3d2a0406,url_809eda7a7e20b3b2,22,5,82,2026-05-09,2026-07-01,keyword article,...,4,2026-05-06,gemini-generate-content,gemini-3-flash-preview,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword_e7cec7ab1804c1c2,url_5fb42bafc4399861,14,3,92,2026-06-15,2026-06-15,keyword article,...,4,2026-06-01,gemini-generate-content,gemini-3-flash-preview,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword_56b0062a1d8b7524,url_ece0abc3e5fb75f9,24,6,98,2026-05-21,2026-06-01,keyword article,...,4,2026-05-12,gemini-generate-content,gemini-3-flash-preview,15776,2552,NaT,NaT,True,False


**Daily performance inspect**

In [10]:
performance_sample = con.sql(f"""
    SELECT *
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/**/*.parquet'
    )
    LIMIT 5
""").df()

performance_sample

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [11]:
performance_sample.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events',
 'month']

**warehouse ka actual date coverage**

In [12]:
months = con.sql(f"""
    SELECT DISTINCT month
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    ORDER BY month
""").df()

months

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month
0,2025-01
1,2025-02
2,2025-03
3,2025-04
4,2025-05
5,2025-06
6,2025-07
7,2025-08
8,2025-09
9,2025-10


**Check rows in month**

In [13]:
monthly_counts = con.sql(f"""
    SELECT
        month,
        COUNT(*) AS rows
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    GROUP BY month
    ORDER BY month
""").df()

monthly_counts

,month,rows
0,2025-01,1297
1,2025-02,75985
2,2025-03,167859
3,2025-04,285114
4,2025-05,349923
5,2025-06,329201
6,2025-07,469794
7,2025-08,704962
8,2025-09,845813
9,2025-10,2165471


In [14]:
content_monthly = con.sql(f"""
    SELECT
        month,
        COUNT(DISTINCT content_hash_id) AS unique_pages,
        COUNT(*) AS performance_rows
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month >= '2025-09'
    GROUP BY month
    ORDER BY month
""").df()

content_monthly

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,unique_pages,performance_rows
0,2025-09,53127,845813
1,2025-10,110339,2165471
2,2025-11,251384,6793825
3,2025-12,254621,7752930
4,2026-01,261984,7890817
5,2026-02,321546,7355108
6,2026-03,331437,9841378
7,2026-04,362172,10424730
8,2026-05,389153,11687376
9,2026-06,409205,11694072


**how many pages have enough consecutive daily data**

In [15]:
coverage = con.sql(f"""
    SELECT
        content_hash_id,
        COUNT(DISTINCT report_date) AS active_days,
        MIN(report_date) AS first_seen,
        MAX(report_date) AS last_seen
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE report_date >= '2025-10-01'
      AND report_date <= '2026-06-30'
    GROUP BY content_hash_id
    LIMIT 100000
""").df()

coverage.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,active_days,first_seen,last_seen
0,content_a39c91fbc1b6232c,273,2025-10-01,2026-06-30
1,content_ea5a133e5c6c598c,273,2025-10-01,2026-06-30
2,content_190cd7001a47883d,273,2025-10-01,2026-06-30
3,content_3b03426e82d5440e,273,2025-10-01,2026-06-30
4,content_58fe39370a1c38d1,273,2025-10-01,2026-06-30


**Modeling windows**




In [16]:
FEATURE_DAYS = 28
FUTURE_DAYS = 28

TRAIN_START = "2025-10-01"
FINAL_CUTOFF = "2026-05-31"

print("Feature window:", FEATURE_DAYS, "days")
print("Future window:", FUTURE_DAYS, "days")
print("Training start:", TRAIN_START)
print("Final cutoff:", FINAL_CUTOFF)

Feature window: 28 days
Future window: 28 days
Training start: 2025-10-01
Final cutoff: 2026-05-31


In [17]:
cutoffs = con.sql("""
    SELECT cutoff_date
    FROM generate_series(
        DATE '2025-10-29',
        DATE '2026-05-31',
        INTERVAL '14 days'
    ) AS t(cutoff_date)
""").df()

cutoffs

,cutoff_date
0,2025-10-29
1,2025-11-12
2,2025-11-26
3,2025-12-10
4,2025-12-24
5,2026-01-07
6,2026-01-21
7,2026-02-04
8,2026-02-18
9,2026-03-04


**Create the cutoff table inside DuckDB**

In [18]:
con.execute("""
CREATE OR REPLACE TEMP TABLE cutoffs AS
SELECT cutoff_date
FROM generate_series(
    DATE '2025-10-29',
    DATE '2026-05-27',
    INTERVAL '14 days'
) AS t(cutoff_date)
""")

print("Cutoff table created.")

Cutoff table created.


In [19]:
con.sql("SELECT * FROM cutoffs").df()

,cutoff_date
0,2025-10-29
1,2025-11-12
2,2025-11-26
3,2025-12-10
4,2025-12-24
5,2026-01-07
6,2026-01-21
7,2026-02-04
8,2026-02-18
9,2026-03-04


**Build the page-level dataset(DATA CHECK)**

In [20]:
# Check that our connection is still alive
print(con.sql("SELECT 1 AS connection_test").df())

   connection_test
0                1


In [21]:
date_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT content_hash_id) AS pages,
        COUNT(DISTINCT report_date) AS days
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE report_date BETWEEN DATE '2025-10-02'
                          AND DATE '2026-06-24'
""").df()

date_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,pages,days
0,73438992,424135,266


### Modeling dataset

The full relevant warehouse data is processed inside DuckDB. The complete
daily performance release is not loaded into pandas because it contains
tens of millions of rows.

For each content page and cutoff date, the analysis creates features from
the previous 28 days and measures outcomes in the following 28 days.

This preserves the temporal structure of the problem and avoids using
future observations as model features.

In [22]:
daily_page = con.sql(f"""
    SELECT
        report_date,
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        AVG(gsc_avg_position) AS avg_position,
        SUM(ga4_sessions) AS sessions,
        SUM(ga4_engaged_sessions) AS engaged_sessions,
        SUM(ga4_total_engagement_sec) AS engagement_sec
    FROM read_parquet(
        '{HF_DATASET}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE report_date BETWEEN DATE '2025-10-02'
                          AND DATE '2026-06-24'
    GROUP BY
        report_date,
        content_hash_id
""")

print("Daily page-level aggregation created.")

Daily page-level aggregation created.


In [23]:
daily_page.limit(5).df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,content_hash_id,impressions,clicks,avg_position,sessions,engaged_sessions,engagement_sec
0,2025-10-02,content_139af221de75d97d,2.0,0.0,19.500000,0.0,0.0,0.0
1,2025-10-02,content_aeb4a91afebfee1b,6.0,0.0,0.833333,0.0,0.0,0.0
2,2025-10-02,content_25c0d05001e1f0cc,4.0,0.0,2.250000,0.0,0.0,0.0
3,2025-10-02,content_7b3f447e49fb12c7,3.0,0.0,21.000000,0.0,0.0,0.0
4,2025-10-02,content_81ef347244c23662,18.0,1.0,11.444444,0.0,0.0,0.0


In [24]:
daily_page_count = con.sql("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT content_hash_id) AS pages,
        COUNT(DISTINCT report_date) AS days
    FROM daily_page
""").df()

daily_page_count

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,pages,days
0,73434732,424135,266


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
